# HRM Workforce Excel Ingestion - Snapshot Logs

Notebook này ingest các file Excel workforce report có dạng:

```text
excel-files/VCS_Workforce Report 2025_28102025.xlsx
```

Trong đó phần `28102025` được hiểu là ngày snapshot `28/10/2025`.

Pipeline giữ nguyên cơ chế tổng thể:

1. Load `.env.minio`
2. Tìm/download các file Excel từ HRM bucket
3. Đọc các sheet khớp pattern
4. Header nằm ở dòng Excel số 2 (`header=1`)
5. Chỉ lấy các cột có trong mapping, thiếu thì bỏ qua
6. Thêm metadata `filename`, `sheet_name`, `snapshot_date`, `snapshot_date_ts`, `crawled_at_ts`
7. Chống trùng theo cặp `filename + sheet_name + resource_name`
8. Gom toàn bộ file vào 3 dataframe/log table
9. Ghi Parquet local
10. Upload lên MinIO raw bucket
11. Tạo bảng qua Trino/Hive SQL

## Sheet pattern mapping

| Sheet pattern | Resource | Raw location |
|---|---|---|
| `Active_*` | `hr_employee_onboard_logs` | `s3a://vcs-raw/hr-raw/hr_employee_onboard_logs` |
| `Out_*` | `hr_employee_resigned_logs` | `s3a://vcs-raw/hr-raw/hr_employee_resigned_logs` |
| `NhuCau_*` | `hr_employee_headcount_logs` | `s3a://vcs-raw/hr-raw/hr_employee_headcount_logs` |

## 1. Import, env, paths

In [1]:
import os
import re
import json
import time
import tempfile
import urllib3
from pathlib import Path
from typing import Any
from urllib.parse import urlparse
from collections import defaultdict

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import load_dotenv
from minio import Minio
from minio.error import S3Error

load_dotenv(".env.minio", override=True)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Nếu notebook đang chạy ở:
# /prefecthq-external-ingestion-mino/ingestions/data_tools
# thì resources nằm ở:
# /prefecthq-external-ingestion-mino/ingestions/resources
RESOURCES_DIR = Path.cwd().parent / "resources"

MAPPING_DIR = RESOURCES_DIR / "mappings"
PARQUET_SCHEMA_DIR = RESOURCES_DIR / "parquet_schema"
HIVE_SQL_DIR = RESOURCES_DIR / "hive_sql"

DEFAULT_DOMAIN = "hr_raw"

# true  = chỉ print SQL, không execute
# false = execute SQL thật ở bước cuối
DRY_RUN_SQL = os.getenv("DRY_RUN_SQL", "true").lower() == "true"

print("Current working dir:", Path.cwd())
print("Resources dir:", RESOURCES_DIR)
print("DRY_RUN_SQL:", DRY_RUN_SQL)

Current working dir: e:\GitLab\crawlers\prefecthq-external-ingestion\ingestions\data_tools
Resources dir: e:\GitLab\crawlers\prefecthq-external-ingestion\ingestions\resources
DRY_RUN_SQL: False


## 2. Resource config

In [2]:
SHEET_PATTERN_MAPPING: dict[str, dict[str, Any]] = {
    "Active_": {
        "domain": "hr_raw",
        "resource_name": "hr_employee_onboard_logs",
        "header_row": 1,  # Excel row 2
    },
    "Out_": {
        "domain": "hr_raw",
        "resource_name": "hr_employee_resigned_logs",
        "header_row": 1,  # Excel row 2
    },
    "NhuCau_": {
        "domain": "hr_raw",
        "resource_name": "hr_employee_headcount_logs",
        "header_row": 1,  # Excel row 2
    },
}

RESOURCE_NAMES = sorted({cfg["resource_name"] for cfg in SHEET_PATTERN_MAPPING.values()})

for prefix, cfg in SHEET_PATTERN_MAPPING.items():
    print(f"{prefix}* -> {cfg['resource_name']}")

print("Resources:", RESOURCE_NAMES)

Active_* -> hr_employee_onboard_logs
Out_* -> hr_employee_resigned_logs
NhuCau_* -> hr_employee_headcount_logs
Resources: ['hr_employee_headcount_logs', 'hr_employee_onboard_logs', 'hr_employee_resigned_logs']


## 3. MinIO/S3 helpers

In [3]:
def env_get(*keys: str, default: str | None = None) -> str | None:
    for key in keys:
        value = os.getenv(key)
        if value:
            return value
    return default


def get_hrm_bucket() -> str:
    return env_get("MINIO_HRM_BUCKET", "MINIO_BUCKET", default="hrm-data")


def get_raw_bucket() -> str:
    return env_get("MINIO_RAW_BUCKET", "MINIO_BUCKET", default="vcs-raw")


def build_minio_client(endpoint: str, access_key: str, secret_key: str) -> Minio:
    if not endpoint:
        raise ValueError("Missing MinIO endpoint")

    parsed = urlparse(endpoint)
    if not parsed.scheme or not parsed.netloc:
        raise ValueError(f"Invalid MinIO endpoint: {endpoint}")

    # Internal/self-signed cert.
    http_client = urllib3.PoolManager(cert_reqs="CERT_NONE")

    return Minio(
        endpoint=parsed.netloc,
        access_key=access_key,
        secret_key=secret_key,
        secure=parsed.scheme == "https",
        http_client=http_client,
    )


def get_hrm_client() -> Minio:
    endpoint = env_get("MINIO_HRM_ENDPOINT", "MINIO_ENDPOINT")
    access_key = env_get("MINIO_HRM_ACCESS_KEY", "MINIO_ACCESS_KEY")
    secret_key = env_get("MINIO_HRM_SECRET_KEY", "MINIO_SECRET_KEY")

    if not endpoint or not access_key or not secret_key:
        raise ValueError("Missing HRM MinIO config")

    return build_minio_client(endpoint, access_key, secret_key)


def get_raw_client() -> Minio:
    endpoint = env_get("MINIO_RAW_ENDPOINT", "MINIO_ENDPOINT")
    access_key = env_get("MINIO_RAW_ACCESS_KEY", "MINIO_ACCESS_KEY")
    secret_key = env_get("MINIO_RAW_SECRET_KEY", "MINIO_SECRET_KEY")

    if not endpoint or not access_key or not secret_key:
        raise ValueError("Missing RAW MinIO config")

    return build_minio_client(endpoint, access_key, secret_key)


def download_hrm_object(object_key: str) -> Path:
    client = get_hrm_client()
    bucket = get_hrm_bucket()
    local_dir = Path(tempfile.mkdtemp(prefix="hrm_workforce_excel_"))
    local_path = local_dir / Path(object_key).name

    client.fget_object(bucket_name=bucket, object_name=object_key, file_path=str(local_path))
    return local_path

## 4. Discover workforce Excel files

Mặc định notebook sẽ list object trong prefix `excel-files/` và chỉ lấy file khớp format:

```text
VCS_Workforce Report <year>_<ddmmyyyy>.xlsx
```

Nếu muốn chạy một danh sách cố định, set `OBJECT_KEYS_OVERRIDE` ở cell dưới.

In [4]:
EXCEL_OBJECT_PREFIX = os.getenv("EXCEL_OBJECT_PREFIX", "excel-files/")
WORKFORCE_OBJECT_REGEX = re.compile(
    r"^excel-files/VCS_Workforce Report \d{4}_(\d{8})\.xlsx$"
)

# Optional: override thủ công danh sách object_key cần xử lý.
# Ví dụ:
# OBJECT_KEYS_OVERRIDE = ["excel-files/VCS_Workforce Report 2025_28102025.xlsx"]
OBJECT_KEYS_OVERRIDE: list[str] = []


def is_workforce_object_key(object_key: str) -> bool:
    return bool(WORKFORCE_OBJECT_REGEX.match(object_key))


def list_workforce_object_keys() -> list[str]:
    if OBJECT_KEYS_OVERRIDE:
        return sorted(set(OBJECT_KEYS_OVERRIDE))

    client = get_hrm_client()
    bucket = get_hrm_bucket()

    object_keys: list[str] = []
    for obj in client.list_objects(bucket, prefix=EXCEL_OBJECT_PREFIX, recursive=True):
        object_key = obj.object_name
        if is_workforce_object_key(object_key):
            object_keys.append(object_key)

    return sorted(set(object_keys))


OBJECT_KEYS = list_workforce_object_keys()

print("Matched object count:", len(OBJECT_KEYS))
for object_key in OBJECT_KEYS:
    print("-", object_key)

Matched object count: 6
- excel-files/VCS_Workforce Report 2025_28102025.xlsx
- excel-files/VCS_Workforce Report 2025_30092025.xlsx
- excel-files/VCS_Workforce Report 2025_31122025.xlsx
- excel-files/VCS_Workforce Report 2026_09032026.xlsx
- excel-files/VCS_Workforce Report 2026_10042026.xlsx
- excel-files/VCS_Workforce Report 2026_21052026.xlsx


## 5. Load mapping/schema/sql

In [5]:
def load_json(path: Path) -> Any:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_column_mapping(domain: str, resource_name: str) -> dict[str, str]:
    return load_json(MAPPING_DIR / domain / f"{resource_name}.json")


def load_parquet_schema(domain: str, resource_name: str) -> Any:
    return load_json(PARQUET_SCHEMA_DIR / domain / f"{resource_name}.json")


def load_create_table_sql(domain: str, resource_name: str) -> str | None:
    path = HIVE_SQL_DIR / domain / f"create_table_{resource_name}.sql"
    if not path.exists():
        return None
    return path.read_text(encoding="utf-8")


for resource_name in RESOURCE_NAMES:
    print(resource_name)
    print("  Mapping:", (MAPPING_DIR / DEFAULT_DOMAIN / f"{resource_name}.json").exists())
    print("  Schema :", (PARQUET_SCHEMA_DIR / DEFAULT_DOMAIN / f"{resource_name}.json").exists())
    print("  SQL    :", (HIVE_SQL_DIR / DEFAULT_DOMAIN / f"create_table_{resource_name}.sql").exists())

hr_employee_headcount_logs
  Mapping: True
  Schema : True
  SQL    : True
hr_employee_onboard_logs
  Mapping: True
  Schema : True
  SQL    : True
hr_employee_resigned_logs
  Mapping: True
  Schema : True
  SQL    : True


## 6. Normalize helpers

In [6]:
def normalize_column_name(col: Any) -> str:
    return re.sub(r"\s+", " ", str(col).strip())


def clean_raw_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalize_column_name(col) for col in df.columns]
    return df


def normalize_schema(schema: Any) -> dict[str, str]:
    """
    Supports:
    1. Spark-style struct:
       {"type": "struct", "fields": [{"name": "...", "dataType": {"type": "string"}}]}
    2. Simple dict:
       {"col": "string"}
    3. List:
       [{"name": "col", "type": "string"}]
    """
    if isinstance(schema, dict) and schema.get("type") == "struct":
        fields = schema.get("fields", [])
        result = {}
        for field in fields:
            data_type = field.get("dataType") or field.get("datatype") or field.get("type")
            if isinstance(data_type, dict):
                logical_type = data_type.get("type", "string")
            else:
                logical_type = data_type or "string"
            result[field["name"]] = str(logical_type).lower()
        return result

    if isinstance(schema, dict):
        return {str(k): str(v).lower() for k, v in schema.items()}

    if isinstance(schema, list):
        result = {}
        for field in schema:
            name = field.get("name")
            logical_type = field.get("type") or field.get("dataType") or field.get("datatype") or "string"
            if isinstance(logical_type, dict):
                logical_type = logical_type.get("type", "string")
            result[name] = str(logical_type).lower()
        return result

    raise TypeError(f"Unsupported schema format: {type(schema)}")


def validate_mapping_no_duplicate_normalized_sources(column_mapping: dict[str, str]) -> None:
    seen: dict[str, str] = {}
    duplicates: list[tuple[str, str, str]] = []

    for source_col in column_mapping.keys():
        normalized = normalize_column_name(source_col)
        if normalized in seen:
            duplicates.append((normalized, seen[normalized], source_col))
        else:
            seen[normalized] = source_col

    if duplicates:
        detail = "\n".join(
            f"normalized={n!r}, first={first!r}, second={second!r}"
            for n, first, second in duplicates
        )
        raise ValueError(f"Duplicate source columns after normalization:\n{detail}")

## 7. Snapshot/date helpers

In [7]:
def extract_snapshot_datetime(object_key: str) -> pd.Timestamp:
    filename = Path(object_key).name
    match = re.search(r"_(\d{8})\.xlsx$", filename)
    if not match:
        raise ValueError(f"Cannot extract snapshot date from object_key: {object_key}")

    raw_date = match.group(1)  # ddmmyyyy, e.g. 28102025
    return pd.to_datetime(raw_date, format="%d%m%Y", errors="raise")


def extract_snapshot_date(object_key: str) -> tuple[str, int]:
    parsed = extract_snapshot_datetime(object_key)
    snapshot_date = parsed.strftime("%d/%m/%Y")
    snapshot_date_ts = int(parsed.timestamp())
    return snapshot_date, snapshot_date_ts


# Quick check
_example = "excel-files/VCS_Workforce Report 2025_28102025.xlsx"
print(_example, "->", extract_snapshot_date(_example))

excel-files/VCS_Workforce Report 2025_28102025.xlsx -> ('28/10/2025', 1761609600)


## 8. Apply mapping and schema

In [8]:
PANDAS_TYPE_MAPPING = {
    "string": "string",
    "str": "string",
    "int": "Int64",
    "integer": "Int64",
    "bigint": "Int64",
    "long": "Int64",
    "int64": "Int64",
    "float": "float64",
    "double": "float64",
    "boolean": "boolean",
    "bool": "boolean",
}


def apply_column_mapping_soft(df: pd.DataFrame, column_mapping: dict[str, str]) -> pd.DataFrame:
    """
    Logic mới:
    - Nếu tên cột trong Excel khớp mapping -> lấy và rename sang target.
    - Nếu cột trong mapping không có ở Excel -> bỏ qua, không raise lỗi.
    - Nếu không có cột nào khớp -> trả về DataFrame rỗng.
    """
    validate_mapping_no_duplicate_normalized_sources(column_mapping)

    df = clean_raw_columns(df)
    normalized_mapping = {
        normalize_column_name(source_col): target_col
        for source_col, target_col in column_mapping.items()
    }

    available_source_cols = [
        col for col in df.columns
        if col in normalized_mapping
    ]

    if not available_source_cols:
        return pd.DataFrame()

    df = df[available_source_cols]
    df = df.rename(columns={
        source_col: normalized_mapping[source_col]
        for source_col in available_source_cols
    })

    return df


def normalize_boolean_series(series: pd.Series) -> pd.Series:
    if str(series.dtype) == "boolean":
        return series

    value = series.astype("string").str.strip().str.lower()
    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
        "y": True,
        "n": False,
        "có": True,
        "không": False,
        "x": True,
    }
    return value.map(mapping).astype("boolean")


def apply_parquet_schema(df: pd.DataFrame, schema: Any) -> pd.DataFrame:
    df = df.copy()
    schema_dict = normalize_schema(schema)

    for col, logical_type in schema_dict.items():
        if col not in df.columns:
            df[col] = pd.NA

        pandas_type = PANDAS_TYPE_MAPPING.get(logical_type, "string")

        if pandas_type == "Int64":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        elif pandas_type == "float64":
            df[col] = pd.to_numeric(df[col], errors="coerce")
        elif pandas_type == "boolean":
            df[col] = normalize_boolean_series(df[col])
        else:
            df[col] = df[col].astype("string")

    return df[list(schema_dict.keys())]


def add_metadata_columns(df: pd.DataFrame, object_key: str, sheet_name: str) -> pd.DataFrame:
    df = df.copy()
    snapshot_date, snapshot_date_ts = extract_snapshot_date(object_key)

    df["filename"] = Path(object_key).name
    df["sheet_name"] = sheet_name
    df["snapshot_date"] = snapshot_date
    df["snapshot_date_ts"] = snapshot_date_ts
    df["crawled_at_ts"] = int(time.time())

    return df

## 9. Structure test

Cell này kiểm tra nhanh mỗi file/sheet nào sẽ được xử lý và map sang resource nào.

In [9]:
def resolve_sheet_config(sheet_name: str) -> dict[str, Any] | None:
    for prefix, config in SHEET_PATTERN_MAPPING.items():
        if sheet_name.startswith(prefix):
            return config
    return None


def test_excel_structure(object_key: str, local_path: Path | None = None) -> pd.DataFrame:
    if local_path is None:
        local_path = download_hrm_object(object_key)

    excel = pd.ExcelFile(local_path)
    rows = []

    for sheet_name in excel.sheet_names:
        config = resolve_sheet_config(sheet_name)
        if config is None:
            rows.append({
                "object_key": object_key,
                "sheet_name": sheet_name,
                "matched": False,
                "resource_name": None,
                "reason": "sheet pattern not matched",
            })
            continue

        rows.append({
            "object_key": object_key,
            "sheet_name": sheet_name,
            "matched": True,
            "resource_name": config["resource_name"],
            "reason": None,
        })

    return pd.DataFrame(rows)


STRUCTURE_RESULTS = []
LOCAL_EXCEL_PATHS: dict[str, Path] = {}

for object_key in OBJECT_KEYS:
    print("Downloading:", object_key)
    local_path = download_hrm_object(object_key)
    LOCAL_EXCEL_PATHS[object_key] = local_path
    STRUCTURE_RESULTS.append(test_excel_structure(object_key, local_path))

if STRUCTURE_RESULTS:
    STRUCTURE_DF = pd.concat(STRUCTURE_RESULTS, ignore_index=True)
else:
    STRUCTURE_DF = pd.DataFrame(columns=["object_key", "sheet_name", "matched", "resource_name", "reason"])

print("Matched sheets:", int(STRUCTURE_DF["matched"].sum()) if not STRUCTURE_DF.empty else 0)
display(STRUCTURE_DF)

Downloading: excel-files/VCS_Workforce Report 2025_28102025.xlsx
Downloading: excel-files/VCS_Workforce Report 2025_30092025.xlsx
Downloading: excel-files/VCS_Workforce Report 2025_31122025.xlsx
Downloading: excel-files/VCS_Workforce Report 2026_09032026.xlsx
Downloading: excel-files/VCS_Workforce Report 2026_10042026.xlsx
Downloading: excel-files/VCS_Workforce Report 2026_21052026.xlsx
Matched sheets: 22


,object_key,sheet_name,matched,resource_name,reason
0,excel-files/VCS_Workforce Report 2025_28102025...,Cover,False,None,sheet pattern not matched
1,excel-files/VCS_Workforce Report 2025_28102025...,Sheet2,False,None,sheet pattern not matched
2,excel-files/VCS_Workforce Report 2025_28102025...,Report,False,None,sheet pattern not matched
3,excel-files/VCS_Workforce Report 2025_28102025...,Sheet1,False,None,sheet pattern not matched
4,excel-files/VCS_Workforce Report 2025_28102025...,Analyst_MHTC Mới,False,None,sheet pattern not matched
...,...,...,...,...,...
86,excel-files/VCS_Workforce Report 2026_21052026...,Data_Vị trí,False,None,sheet pattern not matched
87,excel-files/VCS_Workforce Report 2026_21052026...,Data_Đơn vị,False,None,sheet pattern not matched
88,excel-files/VCS_Workforce Report 2026_21052026...,DB_2026,False,None,sheet pattern not matched
89,excel-files/VCS_Workforce Report 2026_21052026...,Chỉ tiêu Lõi_Khung,False,None,sheet pattern not matched


## 10. Read, normalize, and aggregate full data

In [10]:
PROCESSED_DATA_LIST: dict[str, list[pd.DataFrame]] = defaultdict(list)
PROCESSED_KEYS: set[tuple[str, str, str]] = set()
PROCESS_LOGS: list[dict[str, Any]] = []


def clean_raw_columns_soft(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    normalized_columns: list[str | None] = []
    seen_columns: set[str] = set()
    duplicate_columns: list[dict[str, Any]] = []

    for original_col in df.columns:
        normalized_col = normalize_column_name(original_col)

        if normalized_col in seen_columns:
            normalized_columns.append(None)
            duplicate_columns.append({
                "original_column": original_col,
                "normalized_column": normalized_col,
            })
        else:
            seen_columns.add(normalized_col)
            normalized_columns.append(normalized_col)

    keep_mask = [col is not None for col in normalized_columns]
    kept_columns = [col for col in normalized_columns if col is not None]

    df = df.loc[:, keep_mask]
    df.columns = kept_columns

    if duplicate_columns:
        print("Dropped duplicate columns after normalization:")
        for item in duplicate_columns:
            print(
                f"  original='{item['original_column']}' "
                f"normalized='{item['normalized_column']}'"
            )

    return df


def apply_column_mapping_soft(
    df: pd.DataFrame,
    column_mapping: dict[str, str],
) -> pd.DataFrame:
    df = clean_raw_columns_soft(df)

    normalized_mapping: dict[str, str] = {}

    for source_col, target_col in column_mapping.items():
        normalized_source_col = normalize_column_name(source_col)

        if normalized_source_col not in normalized_mapping:
            normalized_mapping[normalized_source_col] = target_col

    matched_source_cols = [
        source_col
        for source_col in normalized_mapping.keys()
        if source_col in df.columns
    ]

    if not matched_source_cols:
        return pd.DataFrame()

    df = df[matched_source_cols]
    df = df.rename(columns={
        source_col: normalized_mapping[source_col]
        for source_col in matched_source_cols
    })

    return df


def read_one_workforce_sheet(
    object_key: str,
    local_path: Path,
    sheet_name: str,
    config: dict[str, Any],
) -> pd.DataFrame:
    domain = config.get("domain", DEFAULT_DOMAIN)
    resource_name = config["resource_name"]
    header_row = config.get("header_row", 1)

    df = pd.read_excel(
        local_path,
        sheet_name=sheet_name,
        header=header_row,
        dtype=str,
    )
    df = df.dropna(how="all")

    mapping = load_column_mapping(domain, resource_name)
    schema = load_parquet_schema(domain, resource_name)

    df = apply_column_mapping_soft(df, mapping)

    if df.empty:
        return df

    df = add_metadata_columns(
        df,
        object_key=object_key,
        sheet_name=sheet_name,
    )

    df = apply_parquet_schema(df, schema)

    return df


for object_key in OBJECT_KEYS:
    local_path = LOCAL_EXCEL_PATHS.get(object_key)

    if local_path is None:
        local_path = download_hrm_object(object_key)
        LOCAL_EXCEL_PATHS[object_key] = local_path

    excel = pd.ExcelFile(local_path)

    for sheet_name in excel.sheet_names:
        config = resolve_sheet_config(sheet_name)

        if config is None:
            PROCESS_LOGS.append({
                "object_key": object_key,
                "sheet_name": sheet_name,
                "resource_name": None,
                "status": "skipped",
                "reason": "sheet pattern not matched",
                "rows": 0,
            })
            continue

        resource_name = config["resource_name"]
        dedup_key = (
            Path(object_key).name,
            sheet_name,
            resource_name,
        )

        if dedup_key in PROCESSED_KEYS:
            PROCESS_LOGS.append({
                "object_key": object_key,
                "sheet_name": sheet_name,
                "resource_name": resource_name,
                "status": "skipped",
                "reason": "duplicate filename + sheet_name + resource_name",
                "rows": 0,
            })
            continue

        PROCESSED_KEYS.add(dedup_key)

        print(f"[{resource_name}] Reading {object_key} | sheet={sheet_name}")

        try:
            df = read_one_workforce_sheet(
                object_key=object_key,
                local_path=local_path,
                sheet_name=sheet_name,
                config=config,
            )

            rows = len(df)

            if rows == 0:
                PROCESS_LOGS.append({
                    "object_key": object_key,
                    "sheet_name": sheet_name,
                    "resource_name": resource_name,
                    "status": "skipped",
                    "reason": "no matched columns or empty data",
                    "rows": 0,
                })
                continue

            PROCESSED_DATA_LIST[resource_name].append(df)

            PROCESS_LOGS.append({
                "object_key": object_key,
                "sheet_name": sheet_name,
                "resource_name": resource_name,
                "status": "processed",
                "reason": None,
                "rows": rows,
            })

        except Exception as exc:
            PROCESS_LOGS.append({
                "object_key": object_key,
                "sheet_name": sheet_name,
                "resource_name": resource_name,
                "status": "failed",
                "reason": f"{type(exc).__name__}: {exc}",
                "rows": 0,
            })
            raise


PROCESSED_DATA: dict[str, pd.DataFrame] = {
    resource_name: pd.concat(dfs, ignore_index=True)
    for resource_name, dfs in PROCESSED_DATA_LIST.items()
    if dfs
}

PROCESS_LOG_DF = pd.DataFrame(PROCESS_LOGS)

display(PROCESS_LOG_DF)

for resource_name, df in PROCESSED_DATA.items():
    print(f"{resource_name}: {df.shape}")
    display(df.head())

[hr_employee_onboard_logs] Reading excel-files/VCS_Workforce Report 2025_28102025.xlsx | sheet=Active_2025
[hr_employee_headcount_logs] Reading excel-files/VCS_Workforce Report 2025_28102025.xlsx | sheet=NhuCau_2025
[hr_employee_resigned_logs] Reading excel-files/VCS_Workforce Report 2025_28102025.xlsx | sheet=Out_2025
[hr_employee_resigned_logs] Reading excel-files/VCS_Workforce Report 2025_30092025.xlsx | sheet=Out_2025
[hr_employee_headcount_logs] Reading excel-files/VCS_Workforce Report 2025_30092025.xlsx | sheet=NhuCau_2025
[hr_employee_onboard_logs] Reading excel-files/VCS_Workforce Report 2025_30092025.xlsx | sheet=Active_2025
[hr_employee_onboard_logs] Reading excel-files/VCS_Workforce Report 2025_31122025.xlsx | sheet=Active_2025
[hr_employee_resigned_logs] Reading excel-files/VCS_Workforce Report 2025_31122025.xlsx | sheet=Out_2025
[hr_employee_headcount_logs] Reading excel-files/VCS_Workforce Report 2025_31122025.xlsx | sheet=NhuCau_2025
[hr_employee_onboard_logs] Reading ex

c:\Users\giangnth46\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,object_key,sheet_name,resource_name,status,reason,rows
0,excel-files/VCS_Workforce Report 2025_28102025...,Cover,None,skipped,sheet pattern not matched,0
1,excel-files/VCS_Workforce Report 2025_28102025...,Sheet2,None,skipped,sheet pattern not matched,0
2,excel-files/VCS_Workforce Report 2025_28102025...,Report,None,skipped,sheet pattern not matched,0
3,excel-files/VCS_Workforce Report 2025_28102025...,Sheet1,None,skipped,sheet pattern not matched,0
4,excel-files/VCS_Workforce Report 2025_28102025...,Analyst_MHTC Mới,None,skipped,sheet pattern not matched,0
...,...,...,...,...,...,...
86,excel-files/VCS_Workforce Report 2026_21052026...,Data_Vị trí,None,skipped,sheet pattern not matched,0
87,excel-files/VCS_Workforce Report 2026_21052026...,Data_Đơn vị,None,skipped,sheet pattern not matched,0
88,excel-files/VCS_Workforce Report 2026_21052026...,DB_2026,None,skipped,sheet pattern not matched,0
89,excel-files/VCS_Workforce Report 2026_21052026...,Chỉ tiêu Lõi_Khung,None,skipped,sheet pattern not matched,0


hr_employee_onboard_logs: (5536, 118)


,employee_id,employee_name,status,hire_status,employee_group,contract_company_type,branch_name,division_name,n1_group,n2_group,...,filename,crawled_at_ts,level_effective_date_ts,hire_date_viettel_ts,hire_date_vcs_ts,dob_ts,termination_date_ts,sheet_name,snapshot_date,snapshot_date_ts
0,085840,Nguyễn Sơn Hải,Active,<NA>,TDS,TDS,VCS - HN,Khối Lãnh đạo,Ban Giám đốc,Thủ trưởng đơn vị,...,VCS_Workforce Report 2025_28102025.xlsx,1782221255,<NA>,<NA>,<NA>,<NA>,<NA>,Active_2025,28/10/2025,1761609600
1,004043,Trần Anh Quân,Active,<NA>,TDS,TDS,VCS - HN,Khối Lãnh đạo,Ban Giám đốc,Thủ trưởng đơn vị,...,VCS_Workforce Report 2025_28102025.xlsx,1782221255,<NA>,<NA>,<NA>,<NA>,<NA>,Active_2025,28/10/2025,1761609600
2,044621,Nguyễn Anh Tuấn,Active,<NA>,TDS,TDS,VCS - HN,Khối Lãnh đạo,Ban Giám đốc,Thủ trưởng đơn vị,...,VCS_Workforce Report 2025_28102025.xlsx,1782221255,<NA>,<NA>,<NA>,<NA>,<NA>,Active_2025,28/10/2025,1761609600
3,199735,Lê Quang Hà,Active,<NA>,TDS,TDS,VCS - HN,Khối Lãnh đạo,Ban Giám đốc,Thủ trưởng đơn vị,...,VCS_Workforce Report 2025_28102025.xlsx,1782221255,<NA>,<NA>,<NA>,<NA>,<NA>,Active_2025,28/10/2025,1761609600
4,012130,Nguyễn Ngọc Anh,Active,Tuyển mới,TDS,TDS,VCS - HN,Khối Cơ quan,Ban Kiểm soát,Lãnh đạo Ban,...,VCS_Workforce Report 2025_28102025.xlsx,1782221255,<NA>,<NA>,<NA>,<NA>,<NA>,Active_2025,28/10/2025,1761609600


hr_employee_resigned_logs: (768, 72)


,employee_code,full_name,current_status,new_hire_status,employee_object,contract_type,branch,division_n,unit_n_1,department_n_2,...,crawled_at_ts,hire_date_viettel_ts,hire_date_vcs_ts,date_of_birth_ts,resigned_date_ts,level_effective_date_ts,filename,sheet_name,snapshot_date,snapshot_date_ts
0,807594,Phạm Khánh Ly,Nghỉ việc (out chủ động),<NA>,NDS,<NA>,VCS - HN,Khối Cơ quan,Phòng Tổ chức Hành chính,BP Nhân sự,...,1782221257,<NA>,<NA>,<NA>,<NA>,<NA>,VCS_Workforce Report 2025_28102025.xlsx,Out_2025,28/10/2025,1761609600
1,248016,Nguyễn Thị Thúy Hoa,Nghỉ việc (out chủ động),<NA>,TDS,<NA>,VCS - HN,Khối Dịch vụ,Trung tâm Phân tích chia sẻ nguy cơ An ninh mạng,Threat Intelligence,...,1782221257,<NA>,<NA>,<NA>,<NA>,<NA>,VCS_Workforce Report 2025_28102025.xlsx,Out_2025,28/10/2025,1761609600
2,826430,Nguyễn Ngọc Trâm,Nghỉ việc (out chủ động),<NA>,NDS,<NA>,VCS - HN,Khối Cơ quan,Phòng Tổ chức Hành chính,BP Nhân sự,...,1782221257,<NA>,<NA>,<NA>,<NA>,<NA>,VCS_Workforce Report 2025_28102025.xlsx,Out_2025,28/10/2025,1761609600
3,470277,Nguyễn Trung Anh,Nghỉ việc (Fresher/SV/CTV),<NA>,Fresher/SV,<NA>,VCS - HN,Khối Dịch vụ,Trung tâm Giám sát & Phản ứng trên không gian ...,BP Vận hành SOC,...,1782221257,<NA>,<NA>,<NA>,<NA>,<NA>,VCS_Workforce Report 2025_28102025.xlsx,Out_2025,28/10/2025,1761609600
4,829462,Trần Lê Anh Khoa,Nghỉ việc (out chủ động),<NA>,NDS,<NA>,VCS - HCM,Khối Kinh doanh,Phòng Bán hàng,BP Tư vấn giải pháp,...,1782221257,<NA>,<NA>,<NA>,<NA>,<NA>,VCS_Workforce Report 2025_28102025.xlsx,Out_2025,28/10/2025,1761609600


hr_employee_headcount_logs: (8441, 39)


,headcount_plan,headcount_status,division_n,unit_n_1,department_n_2,department_n_3,role_base,specialization,required_level,position_status,...,is_international_business,is_rnd,is_indirect_group,is_business_support,crawled_at_ts,hire_date_vcs_ts,filename,sheet_name,snapshot_date,snapshot_date_ts
0,ĐB 2025,Hiện có,Khối Lãnh đạo,Ban Giám đốc,Thủ trưởng đơn vị,<NA>,Giám đốc,<NA>,N,<NA>,...,<NA>,<NA>,<NA>,<NA>,1782221264,<NA>,VCS_Workforce Report 2026_09032026.xlsx,NhuCau_2025,09/03/2026,1773014400
1,ĐB 2025,Hiện có,Khối Lãnh đạo,Ban Giám đốc,Thủ trưởng đơn vị,<NA>,Phó Giám đốc,<NA>,N,<NA>,...,<NA>,<NA>,<NA>,<NA>,1782221264,<NA>,VCS_Workforce Report 2026_09032026.xlsx,NhuCau_2025,09/03/2026,1773014400
2,ĐB 2025,Hiện có,Khối Lãnh đạo,Ban Giám đốc,Thủ trưởng đơn vị,<NA>,Phó Giám đốc,<NA>,N,<NA>,...,<NA>,<NA>,<NA>,<NA>,1782221264,<NA>,VCS_Workforce Report 2026_09032026.xlsx,NhuCau_2025,09/03/2026,1773014400
3,ĐB 2025,Hiện có,Khối Lãnh đạo,Ban Giám đốc,Thủ trưởng đơn vị,<NA>,Phó Giám đốc,<NA>,N,<NA>,...,<NA>,<NA>,<NA>,<NA>,1782221264,<NA>,VCS_Workforce Report 2026_09032026.xlsx,NhuCau_2025,09/03/2026,1773014400
4,ĐB 2025,Hiện có,Khối Cơ quan,Ban Kiểm soát,Lãnh đạo Ban,<NA>,Trưởng Ban Kiểm soát,<NA>,N-1,<NA>,...,<NA>,<NA>,<NA>,<NA>,1782221264,<NA>,VCS_Workforce Report 2026_09032026.xlsx,NhuCau_2025,09/03/2026,1773014400


## 11. Write Parquet local

In [11]:
def write_parquet_local(df: pd.DataFrame, resource_name: str) -> Path:
    output_dir = Path(tempfile.mkdtemp(prefix="hrm_workforce_parquet_"))
    output_path = output_dir / f"{resource_name}.parquet"

    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, output_path)

    return output_path


LOCAL_PARQUET_FILES: dict[str, Path] = {}

for resource_name, df in PROCESSED_DATA.items():
    parquet_path = write_parquet_local(df, resource_name)
    LOCAL_PARQUET_FILES[resource_name] = parquet_path

    print(f"[{resource_name}] Local parquet:", parquet_path)
    print(f"[{resource_name}] Rows:", len(df))
    print(f"[{resource_name}] Size bytes:", parquet_path.stat().st_size)

[hr_employee_onboard_logs] Local parquet: C:\Users\GIANGN~1\AppData\Local\Temp\hrm_workforce_parquet_p3rjsbge\hr_employee_onboard_logs.parquet
[hr_employee_onboard_logs] Rows: 5536
[hr_employee_onboard_logs] Size bytes: 310235
[hr_employee_resigned_logs] Local parquet: C:\Users\GIANGN~1\AppData\Local\Temp\hrm_workforce_parquet_xh723iug\hr_employee_resigned_logs.parquet
[hr_employee_resigned_logs] Rows: 768
[hr_employee_resigned_logs] Size bytes: 129919
[hr_employee_headcount_logs] Local parquet: C:\Users\GIANGN~1\AppData\Local\Temp\hrm_workforce_parquet_d0iyp70d\hr_employee_headcount_logs.parquet
[hr_employee_headcount_logs] Rows: 8441
[hr_employee_headcount_logs] Size bytes: 120869


## 12. Upload Parquet to raw bucket

In [20]:
def domain_to_s3_prefix(domain: str) -> str:
    return domain.replace("_", "-")


def upload_parquet_to_raw(parquet_path: Path, domain: str, resource_name: str) -> str:
    bucket = get_raw_bucket()
    client = get_raw_client()

    domain_dash = domain_to_s3_prefix(domain)
    object_name = f"{domain_dash}/{resource_name}/{parquet_path.name}"

    client.fput_object(
        bucket_name=bucket,
        object_name=object_name,
        file_path=str(parquet_path),
        content_type="application/octet-stream",
    )

    return f"s3a://{bucket}/{domain_dash}/{resource_name}"


UPLOAD_LOCATIONS: dict[str, str] = {}

for resource_name, parquet_path in LOCAL_PARQUET_FILES.items():
    domain = DEFAULT_DOMAIN
    location = upload_parquet_to_raw(parquet_path, domain, resource_name)
    UPLOAD_LOCATIONS[resource_name] = location
    print(f"[{resource_name}] Uploaded to {location}")

[hr_employee_onboard_logs] Uploaded to s3a://vcs-raw/hr-raw/hr_employee_onboard_logs
[hr_employee_resigned_logs] Uploaded to s3a://vcs-raw/hr-raw/hr_employee_resigned_logs
[hr_employee_headcount_logs] Uploaded to s3a://vcs-raw/hr-raw/hr_employee_headcount_logs


## 13. Execute SQL create table - final step

Ưu tiên dùng file SQL có sẵn tại:

```text
resources/hive_sql/hr_raw/create_table_<resource_name>.sql
```

Nếu chưa có file SQL, notebook sẽ generate SQL cơ bản từ parquet schema.

In [13]:
HIVE_TYPE_MAPPING = {
    "string": "string",
    "str": "string",
    "int": "bigint",
    "integer": "bigint",
    "bigint": "bigint",
    "long": "bigint",
    "int64": "bigint",
    "float": "double",
    "double": "double",
    "boolean": "boolean",
    "bool": "boolean",
}


def schema_to_hive_columns(schema: Any) -> str:
    schema_dict = normalize_schema(schema)
    lines = []
    for col, logical_type in schema_dict.items():
        hive_type = HIVE_TYPE_MAPPING.get(logical_type, "string")
        lines.append(f"  `{col}` {hive_type}")
    return ",\n".join(lines)


def generate_create_table_sql(domain: str, resource_name: str, location: str) -> str:
    schema = load_parquet_schema(domain, resource_name)
    columns_sql = schema_to_hive_columns(schema)
    table_name = f"{domain}.{resource_name}"

    return f"""
CREATE TABLE IF NOT EXISTS {table_name} (
{columns_sql}
)
WITH (
  external_location = '{location}',
  format = 'PARQUET'
)
""".strip()


def build_create_table_sql(domain: str, resource_name: str, location: str) -> str:
    sql = load_create_table_sql(domain, resource_name)
    if sql is not None:
        return sql
    return generate_create_table_sql(domain, resource_name, location)


def execute_sql(sql: str) -> None:
    dry_run = os.getenv("DRY_RUN_SQL", "true").lower() == "true"

    if dry_run:
        print("DRY_RUN_SQL=true, skip execute SQL")
        print(sql)
        return

    import trino

    host = os.getenv("TRINO_HOST")
    user = env_get("TRINO_USERNAME", "TRINO_USER")

    if not host or not user:
        raise ValueError(
            "Missing TRINO_HOST or TRINO_USERNAME/TRINO_USER. "
            "Set DRY_RUN_SQL=true if you only want to print SQL."
        )

    verify = os.getenv("TRINO_SSL_VERIFY", "true").lower() == "true"
    http_scheme = os.getenv("TRINO_HTTP_SCHEME", "https")

    connect_kwargs = dict(
        host=host,
        port=int(os.getenv("TRINO_PORT", "443")),
        user=user,
        catalog=os.getenv("TRINO_CATALOG", "hive"),
        schema=os.getenv("TRINO_SCHEMA", DEFAULT_DOMAIN),
        http_scheme=http_scheme,
    )

    # Với trino-python-client phiên bản mới, verify có thể truyền qua requests_kwargs.
    if http_scheme == "https":
        connect_kwargs["requests_kwargs"] = {"verify": verify}

    conn = trino.dbapi.connect(**connect_kwargs)
    cur = conn.cursor()
    cur.execute(sql)
    cur.fetchall()
    cur.close()
    conn.close()


for resource_name, location in UPLOAD_LOCATIONS.items():
    domain = DEFAULT_DOMAIN
    sql = build_create_table_sql(domain, resource_name, location)

    print("=" * 120)
    print(f"[{resource_name}] Create table SQL")
    execute_sql(sql)

print(f"All done. SQL step handled for {len(UPLOAD_LOCATIONS)} resources.")

[hr_employee_onboard_logs] Create table SQL


TypeError: Connection.__init__() got an unexpected keyword argument 'requests_kwargs'

## 14. Final summary

In [ ]:
print("Processed resources")
print("=" * 120)
for resource_name, df in PROCESSED_DATA.items():
    print(f"{resource_name}: rows={len(df)}, columns={len(df.columns)}")

print("\nUpload locations")
print("=" * 120)
for resource_name, location in UPLOAD_LOCATIONS.items():
    print(resource_name)
    print(location)

print("\nProcess log")
print("=" * 120)
display(PROCESS_LOG_DF)